# Визуализация сегментации SegFormer

Пайплайн: **изображение → SegFormer → маска классов → карта скоростей**.

Главное — раскраска местности по скорости бега (**зелёный = быстро, красный = медленно**).
Дополнительно: сравнение с палитрой классов, оверлей и анимации (морф, проявление классов, sliding-window).

Если чекпоинта нет — берётся GT (`label.png` / `channels.npy` рядом с картинкой).

In [ ]:
from pathlib import Path
import sys
import os

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    if (ROOT.parent / "src").exists():
        ROOT = ROOT.parent
    else:
        raise FileNotFoundError("Не найден src/. Укажите ROOT вручную.")

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT =", ROOT)

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import matplotlib.pyplot as plt
import numpy as np

from src.routing import (
    CLASS_LABELS_RU,
    CLASS_NAMES,
    DEFAULT_SPEEDS_MPS,
    animate_segmentation,
    copy_speeds,
    prepare_segmentation,
    render_segmentation_map,
    render_speed_map,
    show_segmentation,
    show_speed_map,
    speed_legend_handles,
)
from src.utils.config import load_experiment_config
from src.utils.device import get_device

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

## 1. Параметры

Укажите карту и чекпоинт. Для быстрого просмотра без модели подойдёт любой тайл из `dataset/*/tiles/tile_XXXX/`.

In [ ]:
# Полная карта (если есть image.png) или тайл:
IMAGE_PATH = ROOT / "dataset" / "kurakina_dacha_2017_omaps" / "tiles" / "tile_0000" / "image.jpg"
# IMAGE_PATH = ROOT / "dataset" / "kurakina_dacha_2017_omaps" / "image.png"
# IMAGE_PATH = ROOT / "dataset" / "ufa_komsomolsky_omaps" / "image.png"

CONFIG = "configs/quality_segformer.yaml"
CHECKPOINT = ROOT / "checkpoints" / "quality_segformer_b4" / "best.pt"

# Без чекпоинта — GT рядом с картинкой (label.png или channels.npy):
USE_GT = not CHECKPOINT.exists()

SPEEDS = copy_speeds(DEFAULT_SPEEDS_MPS)
OUT_DIR = ROOT / "runs" / "seg_vis"
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert IMAGE_PATH.exists(), IMAGE_PATH
print("image:", IMAGE_PATH)
print("checkpoint:", CHECKPOINT, "(есть)" if CHECKPOINT.exists() else "(нет → USE_GT)")
print("USE_GT:", USE_GT)

## 2. Сегментация

In [ ]:
cfg = load_experiment_config(CONFIG)
device = get_device()
print("device:", device)

model = None
ckpt = None
if not USE_GT:
    from src.infer_fullmap import load_model_from_checkpoint

    assert CHECKPOINT.exists(), f"Нет чекпоинта: {CHECKPOINT}"
    model = load_model_from_checkpoint(CHECKPOINT, cfg, device)
    ckpt = CHECKPOINT

image, label, meta = prepare_segmentation(
    IMAGE_PATH,
    model=model,
    device=device,
    cfg=cfg,
    checkpoint=ckpt,
    use_gt=USE_GT,
    postprocess=True,
    use_cache=True,
)

present = sorted(int(x) for x in np.unique(label))
print(f"image {image.shape[1]}×{image.shape[0]}, classes={len(present)}")
print("классы:", [CLASS_LABELS_RU.get(CLASS_NAMES.get(i, str(i)), str(i)) for i in present])
if meta.has_scale:
    print(f"resolution = {meta.resolution_m_per_px} м/px")

## 3. Карта скоростей (зелёный → красный)

Цвет пикселя = относительная скорость бега по классу местности.
Непроходимое (вода, запреты) — тёмно-серое.

In [ ]:
speed_rgb = render_speed_map(label, SPEEDS)
class_rgb = render_segmentation_map(label)

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
axes[0].imshow(image)
axes[0].set_title("Исходная карта")
axes[1].imshow(class_rgb)
axes[1].set_title("Классы (палитра ISOM)")
axes[2].imshow(speed_rgb)
axes[2].set_title("Скорость: зелёный → красный")
for ax in axes:
    ax.axis("off")

handles = speed_legend_handles(SPEEDS)
axes[2].legend(
    handles=handles,
    loc="upper left",
    fontsize=6,
    framealpha=0.9,
    ncol=1,
    bbox_to_anchor=(1.02, 1.0),
)
plt.tight_layout()
plt.show()

In [ ]:
# Оверлей скорости поверх исходной карты
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
show_segmentation(label, image=image, mode="overlay", alpha=0.45, ax=axes[0], title="Классы (оверлей)")
show_speed_map(label, image=image, speeds=SPEEDS, mode="overlay", alpha=0.55, ax=axes[1], title="Скорость (оверлей)")
plt.tight_layout()
plt.show()

## 4. Доля классов

Столбцы отсортированы по скорости (слева быстрые).

In [ ]:
counts = {int(i): int(c) for i, c in zip(*np.unique(label, return_counts=True))}
total = sum(counts.values())
rows = []
for idx, n in counts.items():
    name = CLASS_NAMES.get(idx, str(idx))
    rows.append(
        {
            "idx": idx,
            "name": name,
            "ru": CLASS_LABELS_RU.get(name, name),
            "share": n / total,
            "speed": float(SPEEDS.get(name, 0.0)),
        }
    )
rows.sort(key=lambda r: (-r["speed"], -r["share"]))

cmap = plt.colormaps["RdYlGn"]
vmax = max((r["speed"] for r in rows if r["speed"] > 0), default=1.0)
colors = []
for r in rows:
    if r["speed"] <= 0:
        colors.append((0.16, 0.16, 0.18))
    else:
        colors.append(cmap(r["speed"] / vmax)[:3])

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar([r["ru"] for r in rows], [r["share"] * 100 for r in rows], color=colors, edgecolor="k", linewidth=0.4)
ax.set_ylabel("% площади")
ax.set_title("Состав сегментации (цвет = скорость)")
ax.tick_params(axis="x", rotation=35, labelsize=8)
plt.tight_layout()
plt.show()

for r in rows:
    spd = "∅" if r["speed"] <= 0 else f"{r['speed']:.2f} м/с"
    print(f"{r['ru']:28s}  {r['share']*100:5.1f}%  {spd}")

## 5. Анимации

Три режима:
1. **morph** — плавный переход от карты к раскраске по скорости
2. **reveal** — классы проявляются от медленных к быстрым
3. **scan** — заливка окнами, как sliding-window инференс SegFormer

In [ ]:
# Кроссфейд: карта → скорость
gif_morph = animate_segmentation(
    image,
    label,
    mode="morph",
    speeds=SPEEDS,
    out_path=OUT_DIR / f"{IMAGE_PATH.parent.name}_speed_morph.gif",
    fps=16,
    max_side=768,
    n_fade=48,
    hold_start=14,
    hold_end=22,
)
print("saved:", gif_morph)

In [ ]:
# Проявление классов по скорости (сначала медленные)
gif_reveal = animate_segmentation(
    image,
    label,
    mode="reveal",
    speeds=SPEEDS,
    out_path=OUT_DIR / f"{IMAGE_PATH.parent.name}_class_reveal.gif",
    fps=14,
    max_side=768,
    frames_per_class=7,
    hold_end=20,
    slow_first=True,
)
print("saved:", gif_reveal)

In [ ]:
# Sliding-window «сканирование» карты
gif_scan = animate_segmentation(
    image,
    label,
    mode="scan",
    speeds=SPEEDS,
    out_path=OUT_DIR / f"{IMAGE_PATH.parent.name}_window_scan.gif",
    fps=12,
    max_side=768,
    window=96,
    step=64,
    hold_end=18,
)
print("saved:", gif_scan)

## 6. Уверенность модели (опционально)

Если загружен чекпоинт — один forward pass на кропе и теплокарта `max(softmax)`.
Тёмные зоны = модель сомневается.

In [ ]:
if model is None:
    print("Нет модели (USE_GT=True) — пропускаем confidence.")
else:
    import torch
    from src.infer_fullmap import predict_tiles_batch

    h, w = image.shape[:2]
    side = min(512, h, w)
    y0 = (h - side) // 2
    x0 = (w - side) // 2
    crop = image[y0 : y0 + side, x0 : x0 + side]
    crop_label = label[y0 : y0 + side, x0 : x0 + side]

    tile = torch.from_numpy(crop).permute(2, 0, 1).unsqueeze(0)
    probs = predict_tiles_batch(model, tile, device, tta=False)[0].cpu().numpy()
    conf = probs.max(axis=0)
    pred = probs.argmax(axis=0).astype(np.uint8)

    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    axes[0].imshow(crop)
    axes[0].set_title("Кроп")
    axes[1].imshow(render_speed_map(pred, SPEEDS))
    axes[1].set_title("Pred → скорость")
    im = axes[2].imshow(conf, cmap="magma", vmin=0.3, vmax=1.0)
    axes[2].set_title(f"Confidence  mean={conf.mean():.2f}")
    fig.colorbar(im, ax=axes[2], fraction=0.046)
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

    # сравнение с GT-кропом, если USE_GT был False, но label с кэша/модели
    agree = (pred == crop_label).mean() * 100
    print(f"совпадение pred vs label на кропе: {agree:.1f}%")